# ATML PA3: private background LoRA4 continuation from verified step 839
Attach the private original Task1 data and private pa3-lora4-step839-restore dataset. Keep this notebook private. Stop any previous draft training/session before launching this as a saved background run. This executes at most 150 additional updates / 1800 training seconds with the exact original recipe. Download and verify the saved output archive before another run. Never treat printed completion as a saved checkpoint.


In [ ]:
# Set only after the previous Kaggle draft is stopped and no training is active.
BATCH_HANDOFF_CONFIRMED = False
assert BATCH_HANDOFF_CONFIRMED, 'Stop the prior draft/session, then confirm handoff before Save & Run All'


In [ ]:
import subprocess, sys, pathlib, json, zipfile, urllib.request
REPO_COMMIT = "e07b5e1548065f5c913a25756d3abd3e421e6666"
work = pathlib.Path("/kaggle/working")
source = work / "pa3-source"
if not source.exists():
    archive = work / "pa3-source.zip"
    urllib.request.urlretrieve(f"https://github.com/sohauv/atml-pa3/archive/{REPO_COMMIT}.zip", archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(work)
    (work / f"atml-pa3-{REPO_COMMIT}").rename(source)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "transformers==4.51.3", "accelerate==1.6.0", "datasets==3.5.0", "numpy==2.2.5"])
print("Pinned source prepared:", REPO_COMMIT)


In [ ]:
# CPU-only input verification; accept Kaggle automatic ZIP extraction.
import sys, pathlib, zipfile, hashlib, json
sys.path.insert(0, str(source))
from scripts.kaggle_task1 import unpack_inputs
archives = list(pathlib.Path("/kaggle/input").rglob("pa3-task1-input.zip"))
if len(archives) == 1:
    input_archive = archives[0]
else:
    candidates = [p.parent for p in pathlib.Path("/kaggle/input").rglob("metadata.json") if (p.parent / "train.jsonl").is_file() and (p.parent / "source_ids.json").is_file()]
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one original Task1 input; found {len(candidates)}")
    original = candidates[0]
    actual = hashlib.sha256((original / "train.jsonl").read_bytes()).hexdigest()
    if actual != json.loads((original / "metadata.json").read_text())["sha256"]:
        raise RuntimeError("Original training checksum mismatch")
    input_archive = work / "pa3-task1-input.zip"
    if not input_archive.exists():
        with zipfile.ZipFile(input_archive, "x", zipfile.ZIP_DEFLATED) as z:
            for name in ("train.jsonl", "metadata.json", "source_ids.json"):
                z.write(original / name, "task1/" + name)
data_dir = unpack_inputs(input_archive, work / "pa3/data")
print("Original training data checksum verified:", data_dir)


In [ ]:
import pathlib, json, shutil, hashlib
EXPECTED = {'lora4-recovery/config.json': '3ee705d1377c6d0b9aae5dadcd0c5f1cbe28e39674e4a398c0befde93d6035f4', 'lora4-recovery/provenance.json': '74a689ea86d5aa90bb068463bc45db8502c5450cf6519879ef7344ac60450691', 'lora4-recovery/sessions.jsonl': '96e876bf2e8b1c6bc31abf678b3801b97460df0a2042db33acda7800d24e239e', 'lora4-recovery/status.json': 'a5815596c46dc7c86cf836b5cb482511964e4246a09e6204ef6f2a023d455d81', 'lora4-recovery/train.jsonl': '6b3db29006f083ac3bc11d4ef36008751e35096ae986e3e6241d411e9a7ae1b2', 'lora4-recovery/resume-0839-1793399575007780443/recovery.pt': '32e7ae5172e54a7a7989c82b3ed0d12e132f6fd349177bc1864438326fade02d', 'lora4-recovery/resume-0839-1793399575007780443/summary.json': '202ad25120ad8b3ad9d1d11472a173cdb4cb22a02056d7a80f863ce12856840b', 'lora4-recovery/resume-0839-1793399575007780443/complete.json': 'f3287d3aa622281706348da7fb154f161aa9678a85d81784958c8f5c07352974', 'lora4-recovery/resume-0839-1793399575007780443/projected_factors.pt': '6512298fed6a26c1e39eb072f20a3487074b06e151fd495075bde5b2e521ee33', 'lora4-recovery/step-0312/recovery.pt': 'f7f2e6d591f0fbe8e055eb977891c9029ad5cd52eb1fb4df9aa16764f5284fbf', 'lora4-recovery/step-0312/summary.json': 'b862e082fc47eb44966860878a57c81a34f43be39043310bb62ed16f254a878a', 'lora4-recovery/step-0312/complete.json': '01a2e607013190061a51be120b9d5e50f42967293967c6e063b3f588acefc929', 'lora4-recovery/step-0312/projected_factors.pt': '6d6dc3e12a697121892fdd3168cebd0962eb584202bcd60ec387aa84bf440b43', 'lora4-recovery/step-0625/recovery.pt': '366dac8aaa3adbdf69d418e76ffe7d4736ab7e805e921ad07d7b831fa7c7cd55', 'lora4-recovery/step-0625/summary.json': 'b6983228a82eafe1a0e4013e25f96446da6051ea3e77f899a40729ea812d4d1b', 'lora4-recovery/step-0625/complete.json': '9a9601646244d05c6edde0165f7e76cbe1ab22ab9b202b1486bcbcb6e7d6c0bd', 'lora4-recovery/step-0625/projected_factors.pt': 'd72e35d30956a196cc2600ab371c2c8362f8bf0de6db078579fb2588e349210e'}
roots = [p.parent.parent for p in pathlib.Path('/kaggle/input').rglob('complete.json') if p.parent.name == 'resume-0839-1793399575007780443']
assert len(roots) == 1, 'Attach the private pa3-lora4-step839-restore input exactly once'
root = roots[0]
run = work / 'pa3/runs/lora4-kaggle-20261030'
assert not run.exists(), 'Fresh background job required; refusing to overwrite a run'
for relative, digest in EXPECTED.items():
    relative = pathlib.PurePosixPath(relative).relative_to('lora4-recovery')
    file = root / str(relative)
    assert file.is_file() and hashlib.file_digest(file.open('rb'), 'sha256').hexdigest() == digest, str(file)
shutil.copytree(root, run)
checkpoint = run / 'resume-0839-1793399575007780443'
summary = json.loads((checkpoint / 'summary.json').read_text())
assert summary['update'] == 839 and summary['successful_optimizer_updates'] == 839 and summary['skipped_updates'] == 0
for step in (312, 625):
    assert (run / ('step-%04d' % step) / 'complete.json').is_file()
print('VERIFIED_RESTORE', summary, flush=True)


In [ ]:
import subprocess, sys, json, zipfile, hashlib
command = [sys.executable, '-m', 'scripts.kaggle_task1', '--archive', str(input_archive), '--method', 'lora4', '--run-id', 'lora4-kaggle-20261030', '--seconds', '1800', '--max-updates', '150', '--resume', str(checkpoint), '--handoff-verified']
print('START_BOUNDED_RECOVERY_FROM', checkpoint, flush=True)
try:
    subprocess.check_call(command, cwd=source)
finally:
    status = json.loads((run / 'status.json').read_text())
    saved = run / pathlib.Path(status['checkpoint']).name
    assert (saved / 'complete.json').is_file()
    result = json.loads((saved / 'summary.json').read_text())
    assert 839 <= result['update'] <= 989
    archive = work / ('lora4-backup-step-%04d.zip' % result['update'])
    with zipfile.ZipFile(archive, 'x', zipfile.ZIP_DEFLATED) as backup:
        for file in run.rglob('*'):
            if file.is_file():
                backup.write(file, str(pathlib.Path('lora4-recovery') / file.relative_to(run)))
    with zipfile.ZipFile(archive) as backup:
        assert backup.testzip() is None
    evidence = {'status': status, 'summary': result, 'archive': str(archive), 'archive_bytes': archive.stat().st_size, 'archive_sha256': hashlib.file_digest(archive.open('rb'), 'sha256').hexdigest()}
    (work / 'checkpoint-backup-evidence.json').write_text(json.dumps(evidence, indent=2))
    print('DOWNLOAD_AND_VERIFY_BEFORE_NEXT_RUN', json.dumps(evidence), flush=True)
